In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

# Set the styling for visualizations
plt.style.use('ggplot')
sns.set_palette("Set2")
sns.set_context("notebook", font_scale=1.2)

# Load the dataset
df = pd.read_csv('data/financial_ai_dataset.csv')

# 1. Data Exploration and Preprocessing
# =====================================

# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Columns:", df.columns.tolist())
print("\nSample Data:")
print(df.head())

# Check for missing values
print("\nMissing Values:", df.isnull().sum().sum())

# Basic statistical summary
print("\nBasic Statistics:")
print(df.describe())

# Convert categorical data if needed
categorical_cols = ['education_level', 'income_level', 'rec_type', 'risk_level']
for col in categorical_cols:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col])

Dataset Shape: (500, 26)

Dataset Columns: ['age', 'income_level', 'education_level', 'financial_literacy', 'risk_tolerance', 'app_usage_frequency', 'time_reviewing_recs', 'follow_through_rate', 'override_frequency', 'savings_rate_change', 'investment_returns', 'budget_adherence', 'debt_reduction', 'net_worth_growth', 'trust_score', 'perceived_usefulness', 'perceived_accuracy', 'rec_type', 'risk_level', 'confidence_score', 'user_satisfaction', 'continued_usage', 'explanation_quality', 'transparency_rating', 'privacy_concerns', 'financial_outcomes']

Sample Data:
   age  income_level  education_level  financial_literacy  risk_tolerance  \
0   56             5                1                   2        0.463494   
1   69             5                5                   3        0.379786   
2   46             5                1                   2        0.863334   
3   32             3                1                   2        0.519082   
4   60             4                1         

In [2]:
# 2. Feature Engineering and Analysis
# ==================================

# Create new composite features
df['trust_efficiency_ratio'] = df['trust_score'] / (df['time_reviewing_recs'] + 1)  # Adding 1 to avoid division by zero
df['financial_outcome_composite'] = (df['savings_rate_change'] + 
                                     df['investment_returns']/20 + 
                                     df['debt_reduction'] + 
                                     df['net_worth_growth']) / 4

df['recommendation_acceptance'] = 1 - df['override_frequency']
df['trust_perception_gap'] = df['trust_score'] - ((df['perceived_usefulness'] + df['perceived_accuracy']) * 10)

# 3. Exploratory Data Analysis
# ===========================

# Setup EDA notebook section with visualizations
plt.figure(figsize=(10, 6))
sns.histplot(df['trust_score'], kde=True)
plt.title('Distribution of Trust Scores')
plt.xlabel('Trust Score')
plt.savefig('trust_score_distribution.png')
plt.close()

plt.figure(figsize=(12, 8))
correlation = df.corr()
mask = np.triu(correlation)
sns.heatmap(correlation, annot=False, mask=mask, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()

In [3]:
# 4. Trust Analysis
# ================

# Examine factors affecting trust
trust_factors = ['financial_literacy', 'app_usage_frequency', 'explanation_quality', 
                 'transparency_rating', 'privacy_concerns', 'override_frequency']

plt.figure(figsize=(14, 10))
for i, factor in enumerate(trust_factors, 1):
    plt.subplot(3, 2, i)
    sns.scatterplot(x=df[factor], y=df['trust_score'], alpha=0.7)
    plt.title(f'Trust Score vs {factor}')
    plt.tight_layout()
plt.savefig('trust_factors.png')
plt.close()

# Group by age ranges and calculate mean trust scores
df['age_group'] = pd.cut(df['age'], bins=[18, 30, 45, 60, 100], labels=['18-30', '31-45', '46-60', '60+'])
trust_by_age = df.groupby('age_group')['trust_score'].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='age_group', y='trust_score', data=trust_by_age)
plt.title('Average Trust Score by Age Group')
plt.ylabel('Average Trust Score')
plt.xlabel('Age Group')
plt.savefig('trust_by_age.png')
plt.close()

In [4]:
# 5. Efficiency Analysis
# ====================

# Analyze relationship between efficiency metrics and financial outcomes
efficiency_metrics = ['time_reviewing_recs', 'follow_through_rate', 'override_frequency']
financial_outcomes = ['savings_rate_change', 'investment_returns', 'budget_adherence', 'debt_reduction', 'net_worth_growth']

# Efficiency correlation matrix
efficiency_corr = df[efficiency_metrics + financial_outcomes].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(efficiency_corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation between Efficiency Metrics and Financial Outcomes')
plt.tight_layout()
plt.savefig('efficiency_correlation.png')
plt.close()

# Scatter plot of follow-through rate vs financial outcomes
plt.figure(figsize=(12, 8))
for i, outcome in enumerate(financial_outcomes, 1):
    plt.subplot(2, 3, i)
    sns.scatterplot(x='follow_through_rate', y=outcome, data=df, alpha=0.7, hue='risk_tolerance')
    plt.title(f'Follow-through Rate vs {outcome}')
plt.tight_layout()
plt.savefig('follow_through_outcomes.png')
plt.close()

In [5]:
# 6. User Segmentation
# ===================

# Perform clustering to identify user segments based on trust and behavior
# Select features for clustering
clustering_features = ['trust_score', 'financial_literacy', 'risk_tolerance', 
                       'follow_through_rate', 'override_frequency', 'app_usage_frequency']

# Scale the data
X_cluster = df[clustering_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Determine optimal number of clusters using the elbow method
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.savefig('elbow_method.png')
plt.close()

# Apply K-means with the chosen number of clusters (4 in this case)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze cluster characteristics
cluster_profile = df.groupby('cluster')[clustering_features + ['financial_outcome_composite']].mean()
print("\nCluster Profiles:")
print(cluster_profile)

# Visualize clusters using PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
df['pca1'] = pca_result[:, 0]
df['pca2'] = pca_result[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(x='pca1', y='pca2', hue='cluster', data=df, palette='viridis', s=100, alpha=0.7)
plt.title('User Segments Visualization')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.savefig('user_segments.png')
plt.close()


Cluster Profiles:
         trust_score  financial_literacy  risk_tolerance  follow_through_rate  \
cluster                                                                         
0          59.496063            4.236220        0.429014             0.343602   
1          46.794521            3.198630        0.435089             0.694205   
2          51.145455            2.354545        0.719213             0.664880   
3          48.726496            1.982906        0.447996             0.278079   

         override_frequency  app_usage_frequency  financial_outcome_composite  
cluster                                                                        
0                  0.411532             2.669291                     0.211773  
1                  0.677609             5.609589                     0.206208  
2                  0.545554             2.236364                     0.210065  
3                  0.332252             5.068376                     0.205798  


In [6]:
# 7. Predictive Modeling
# =====================

# Build models to predict trust scores and financial outcomes

# Prepare features and target variables
X = df.drop(['trust_score', 'financial_outcome_composite', 'pca1', 'pca2', 'cluster', 'age_group'], axis=1)
y_trust = df['trust_score']
y_outcome = df['financial_outcome_composite']

# Split data into training and testing sets
X_train, X_test, y_trust_train, y_trust_test = train_test_split(X, y_trust, test_size=0.2, random_state=42)
_, _, y_outcome_train, y_outcome_test = train_test_split(X, y_outcome, test_size=0.2, random_state=42)

# Trust Score prediction model
print("\nTraining Trust Score Prediction Model...")
rf_trust = RandomForestRegressor(n_estimators=100, random_state=42)
rf_trust.fit(X_train, y_trust_train)

# Financial outcome prediction model
print("Training Financial Outcome Prediction Model...")
gb_outcome = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_outcome.fit(X_train, y_outcome_train)

# Evaluate models
y_trust_pred = rf_trust.predict(X_test)
y_outcome_pred = gb_outcome.predict(X_test)

print("\nTrust Score Model Performance:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_trust_test, y_trust_pred)):.4f}")
print(f"R²: {r2_score(y_trust_test, y_trust_pred):.4f}")
print("Accuracy:", rf_trust.score(X_test, y_trust_test))

print("\nFinancial Outcome Model Performance:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_outcome_test, y_outcome_pred)):.4f}")
print(f"R²: {r2_score(y_outcome_test, y_outcome_pred):.4f}")
print("Accuracy:", gb_outcome.score(X_test, y_outcome_test))

# Feature importance for trust model
trust_importance = permutation_importance(rf_trust, X_test, y_trust_test, n_repeats=10, random_state=42)
trust_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': trust_importance.importances_mean
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=trust_importance_df.head(15))
plt.title('Feature Importance for Trust Score Prediction')
plt.tight_layout()
plt.savefig('trust_feature_importance.png')
plt.close()

# Feature importance for financial outcome model
outcome_importance = permutation_importance(gb_outcome, X_test, y_outcome_test, n_repeats=10, random_state=42)
outcome_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': outcome_importance.importances_mean
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=outcome_importance_df.head(15))
plt.title('Feature Importance for Financial Outcome Prediction')
plt.tight_layout()
plt.savefig('outcome_feature_importance.png')
plt.close()



Training Trust Score Prediction Model...
Training Financial Outcome Prediction Model...

Trust Score Model Performance:
RMSE: 7.9537
R²: 0.9155
Accuracy: 0.9155355589056382

Financial Outcome Model Performance:
RMSE: 0.0188
R²: 0.9789
Accuracy: 0.9788551927184324


In [7]:
# 8. Ethics and Privacy Analysis
# ============================

# Analyze how privacy concerns impact trust and usage
privacy_impact = df.groupby('privacy_concerns')[['trust_score', 'app_usage_frequency', 'follow_through_rate']].mean()
print("\nImpact of Privacy Concerns:")
print(privacy_impact)

plt.figure(figsize=(10, 6))
privacy_impact.plot(kind='bar')
plt.title('Impact of Privacy Concerns on Trust and Usage')
plt.xlabel('Privacy Concern Level')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('privacy_impact.png')
plt.close()

# Analyze relationship between transparency and trust
plt.figure(figsize=(10, 6))
sns.boxplot(x='transparency_rating', y='trust_score', data=df)
plt.title('Relationship Between Transparency and Trust')
plt.xlabel('Transparency Rating')
plt.ylabel('Trust Score')
plt.savefig('transparency_trust.png')
plt.close()

# 9. Recommendation System Evaluation
# =================================

# Analyze effectiveness of different recommendation types
rec_type_analysis = df.groupby('rec_type')[['follow_through_rate', 'financial_outcome_composite', 'user_satisfaction']].mean()
print("\nRecommendation Type Analysis:")
print(rec_type_analysis)

plt.figure(figsize=(12, 6))
rec_type_analysis.plot(kind='bar')
plt.title('Performance by Recommendation Type')
plt.xlabel('Recommendation Type')
plt.ylabel('Average Value')
plt.xticks(rotation=0)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('recommendation_type_analysis.png')
plt.close()


Impact of Privacy Concerns:
                  trust_score  app_usage_frequency  follow_through_rate
privacy_concerns                                                       
1                   51.040000             4.210000             0.483655
2                   52.990476             4.228571             0.460112
3                   54.450980             3.921569             0.540479
4                   49.292135             3.685393             0.493806
5                   49.096154             3.884615             0.527965

Recommendation Type Analysis:
          follow_through_rate  financial_outcome_composite  user_satisfaction
rec_type                                                                     
0                    0.503364                     0.203235           3.129630
1                    0.523401                     0.199833           2.913043
2                    0.484444                     0.218430           3.160000


<Figure size 1000x600 with 0 Axes>

<Figure size 1200x600 with 0 Axes>

In [8]:
# 10. Insights and Recommendations
# =============================

# Prepare a summary of key findings
print("\n======================================")
print("Key Insights and Recommendations")
print("======================================")

# Trust factors analysis
print("\n1. Trust Factors Analysis:")
top_trust_factors = trust_importance_df.head(5)
print(f"Top factors influencing trust: {', '.join(top_trust_factors['Feature'].tolist())}")

# User segments analysis
print("\n2. User Segment Analysis:")
for cluster in range(n_clusters):
    segment_size = (df['cluster'] == cluster).sum()
    segment_pct = segment_size / len(df) * 100
    avg_trust = df[df['cluster'] == cluster]['trust_score'].mean()
    avg_outcome = df[df['cluster'] == cluster]['financial_outcome_composite'].mean()
    print(f"Segment {cluster+1} ({segment_size} users, {segment_pct:.1f}%): Avg Trust = {avg_trust:.1f}, Avg Financial Outcome = {avg_outcome:.3f}")

# Recommendation effectiveness
print("\n3. Recommendation Effectiveness:")
best_rec = rec_type_analysis['financial_outcome_composite'].idxmax()
print(f"Most effective recommendation type: {best_rec}")

# Privacy and transparency
print("\n4. Privacy and Transparency:")
trust_diff = privacy_impact.loc[1]['trust_score'] - privacy_impact.loc[5]['trust_score']
print(f"Trust score difference between lowest and highest privacy concerns: {trust_diff:.2f}")

# Generate strategic recommendations
print("\n5. Strategic Recommendations:")
print("a. Improve transparency to boost trust and adoption")
print("b. Target specific user segments with tailored communication strategies")
print("c. Optimize recommendation algorithms based on user risk profiles")
print("d. Address privacy concerns through enhanced security features and clear communication")
print("e. Focus on explaining recommendations to improve follow-through rates")



Key Insights and Recommendations

1. Trust Factors Analysis:
Top factors influencing trust: trust_perception_gap, trust_efficiency_ratio, perceived_usefulness, time_reviewing_recs, perceived_accuracy

2. User Segment Analysis:
Segment 1 (127 users, 25.4%): Avg Trust = 59.5, Avg Financial Outcome = 0.212
Segment 2 (146 users, 29.2%): Avg Trust = 46.8, Avg Financial Outcome = 0.206
Segment 3 (110 users, 22.0%): Avg Trust = 51.1, Avg Financial Outcome = 0.210
Segment 4 (117 users, 23.4%): Avg Trust = 48.7, Avg Financial Outcome = 0.206

3. Recommendation Effectiveness:
Most effective recommendation type: 2

4. Privacy and Transparency:
Trust score difference between lowest and highest privacy concerns: 1.94

5. Strategic Recommendations:
a. Improve transparency to boost trust and adoption
b. Target specific user segments with tailored communication strategies
c. Optimize recommendation algorithms based on user risk profiles
d. Address privacy concerns through enhanced security features a

In [9]:
# 11. Model Implementation Plan
# ==========================

print("\n======================================")
print("Implementation Plan")
print("======================================")

print("\n1. Trust Enhancement Module:")
print("   - Deploy trust prediction model to identify at-risk users")
print("   - Implement targeted transparency improvements for users with low trust scores")

print("\n2. Personalized Recommendation System:")
print("   - Use cluster analysis to create segment-specific recommendation strategies")
print("   - Adjust recommendation confidence thresholds based on user risk tolerance")

print("\n3. User Experience Optimization:")
print("   - Reduce friction in high-impact recommendation categories")
print("   - Implement A/B testing to measure impact of UI changes on trust scores")

print("\n4. Ethics and Privacy Framework:")
print("   - Establish continuous monitoring for recommendation bias")
print("   - Create transparency metrics dashboard for internal and external stakeholders")

print("\n5. Measurement and Evaluation:")
print("   - Track changes in trust and financial outcomes over time")
print("   - Measure ROI of AI-assisted recommendations vs traditional advisory")

# 12. Future Research Directions
# ===========================

print("\n======================================")
print("Future Research Directions")
print("======================================")

print("\n1. Longitudinal Analysis:")
print("   - Track how trust and financial outcomes evolve over longer time periods")
print("   - Study the effect of major financial events on AI tool adoption")

print("\n2. Behavioral Economics:")
print("   - Explore nudge techniques to improve recommendation follow-through")
print("   - Study cognitive biases that affect user interaction with AI finance tools")

print("\n3. Advanced AI Methods:")
print("   - Test reinforcement learning approaches for recommendation optimization")
print("   - Develop explainable AI methods tailored to financial decision-making")

print("\n4. Cross-Cultural Analysis:")
print("   - Examine how cultural differences affect trust in AI financial advisors")
print("   - Develop culture-specific trust building strategies")


Implementation Plan

1. Trust Enhancement Module:
   - Deploy trust prediction model to identify at-risk users
   - Implement targeted transparency improvements for users with low trust scores

2. Personalized Recommendation System:
   - Use cluster analysis to create segment-specific recommendation strategies
   - Adjust recommendation confidence thresholds based on user risk tolerance

3. User Experience Optimization:
   - Reduce friction in high-impact recommendation categories
   - Implement A/B testing to measure impact of UI changes on trust scores

4. Ethics and Privacy Framework:
   - Establish continuous monitoring for recommendation bias
   - Create transparency metrics dashboard for internal and external stakeholders

5. Measurement and Evaluation:
   - Track changes in trust and financial outcomes over time
   - Measure ROI of AI-assisted recommendations vs traditional advisory

Future Research Directions

1. Longitudinal Analysis:
   - Track how trust and financial outcom

In [10]:
#Sample User
def predict_trust_and_recommendations(user_data):
    """
    Function to predict trust score and generate personalized recommendations
    
    Parameters:
    user_data (dict): Dictionary containing user features
    
    Returns:
    dict: Prediction results and personalized recommendations
    """
    # Create a DataFrame for prediction with only the features needed for the models
    user_df_for_prediction = pd.DataFrame(columns=X.columns)
    user_df_for_prediction.loc[0] = 0  # Initialize with zeros
    
    # Fill in available values
    for column in X.columns:
        if column in user_data:
            user_df_for_prediction.loc[0, column] = user_data[column]
    
    # Add derived features if they were in the training data
    if 'trust_efficiency_ratio' in X.columns:
        user_df_for_prediction['trust_efficiency_ratio'] = user_data['trust_score'] / (user_data['time_reviewing_recs'] + 1)
    
    if 'recommendation_acceptance' in X.columns:
        user_df_for_prediction['recommendation_acceptance'] = 1 - user_data['override_frequency']
    
    # Make predictions
    predicted_trust = rf_trust.predict(user_df_for_prediction)[0]
    predicted_outcome = gb_outcome.predict(user_df_for_prediction)[0]
    
    # Create a separate DataFrame for clustering that includes all original user data
    user_df_full = pd.DataFrame([user_data])
    
    # Determine user segment - make sure clustering_features are all in user_df_full
    # First, check if all clustering features are available
    missing_features = [feat for feat in clustering_features if feat not in user_df_full.columns]
    if missing_features:
        print(f"Warning: Missing features for clustering: {missing_features}")
        # You can either add missing features with default values, or skip clustering
        for feat in missing_features:
            user_df_full[feat] = 0  # Default value
    
    user_features = user_df_full[clustering_features].copy()
    user_scaled = scaler.transform(user_features)
    user_cluster = kmeans.predict(user_scaled)[0]
    
    # Rest of your function remains the same
    # Generate recommendations based on user segment and risk tolerance
    if user_data['risk_tolerance'] > 0.7:
        rec_strategy = "Aggressive growth strategy with comprehensive explanations"
    elif user_data['risk_tolerance'] > 0.4:
        rec_strategy = "Balanced approach with medium-risk opportunities"
    else:
        rec_strategy = "Conservative strategy focused on stability and incremental growth"
    
    # Adjust based on trust score
    if predicted_trust < 30:
        transparency_level = "High transparency with detailed explanations for all recommendations"
    else:
        transparency_level = "Standard transparency with option to view detailed explanations"
    
    return {
        'predicted_trust': predicted_trust,
        'predicted_financial_outcome': predicted_outcome,
        'user_segment': user_cluster,
        'recommended_strategy': rec_strategy,
        'transparency_approach': transparency_level
    }

# Example usage of the prediction function
sample_user = {
    'age': 35,
    'income_level': 4,
    'education_level': 3,
    'financial_literacy': 3,
    'risk_tolerance': 0.5,
    'app_usage_frequency': 5,
    'time_reviewing_recs': 15.0,
    'follow_through_rate': 0.7,
    'override_frequency': 0.3,
    'trust_score': 50,  # Initial or survey-based value
    'perceived_usefulness': 3,
    'perceived_accuracy': 4,
    'explanation_quality': 3,
    'transparency_rating': 4,
    'privacy_concerns': 2,
    'rec_type': 1,  # Balanced
    'risk_level': 3
}

print("\n======================================")
print("Sample Prediction for New User")
print("======================================")
prediction_result = predict_trust_and_recommendations(sample_user)
for key, value in prediction_result.items():
    print(f"{key}: {value}")

print("\n======================================")
print("Conclusion")
print("======================================")
print("This analysis demonstrates that AI-assisted financial tools can significantly impact user financial outcomes,")
print("with trust and transparency being critical factors for success. By focusing on personalized experiences,")
print("clear explanations, and addressing privacy concerns, these platforms can improve both user trust and financial efficiency.")


Sample Prediction for New User
predicted_trust: 76.72
predicted_financial_outcome: 0.03761119421140269
user_segment: 1
recommended_strategy: Balanced approach with medium-risk opportunities
transparency_approach: Standard transparency with option to view detailed explanations

Conclusion
This analysis demonstrates that AI-assisted financial tools can significantly impact user financial outcomes,
with trust and transparency being critical factors for success. By focusing on personalized experiences,
clear explanations, and addressing privacy concerns, these platforms can improve both user trust and financial efficiency.
